In [123]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import os 
from dotenv import load_dotenv
import requests

# Load environment variables from .env file
load_dotenv()

# Retrieve API key from environment
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

# Initialize Groq Chat Model
model = ChatGroq(
    model="llama-3.1-8b-instant"
)


In [124]:
#tool create


@tool
def multiply(a : int, b : int) -> int:
    """Multiplies two numbers."""
    return a * b

print(multiply.invoke({'a': 3,'b':4}))  # Output: 12  


12


In [125]:
#Tool Binding
llm_with_tool = model.bind_tools([multiply])
llm_with_tool

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000231E0404FC0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000231E0405940>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'multiply', 'description': 'Multiplies two numbers.', 'parameters': {'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a', 'b'], 'type': 'object'}}}]}, config={}, config_factories=[])

In [126]:
#Tool calling
result = llm_with_tool.invoke("hi how are you?")
result



AIMessage(content="I'm functioning properly, thanks for asking. How can I assist you today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 219, 'total_tokens': 236, 'completion_time': 0.03182155, 'completion_tokens_details': None, 'prompt_time': 0.098006449, 'prompt_tokens_details': None, 'queue_time': 0.053421091, 'total_time': 0.129827999}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e1a99-3917-7e73-988d-36d70daad753-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 219, 'output_tokens': 17, 'total_tokens': 236})

In [127]:
query = HumanMessage("What is 3 multiplied by 4?")

In [128]:
messages = [query]
messages

[HumanMessage(content='What is 3 multiplied by 4?', additional_kwargs={}, response_metadata={})]

In [129]:
result = llm_with_tool.invoke(messages)

messages.append(result)
messages

[HumanMessage(content='What is 3 multiplied by 4?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'kf8fhgf81', 'function': {'arguments': '{"a":3,"b":4}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 223, 'total_tokens': 242, 'completion_time': 0.042393343, 'completion_tokens_details': None, 'prompt_time': 0.016905001, 'prompt_tokens_details': None, 'queue_time': 0.059704289, 'total_time': 0.059298344}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e1a99-3b32-7030-815d-50661a640209-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 4}, 'id': 'kf8fhgf81', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 223, 'output_tokens': 19, 'total_tokens': 242})]

In [130]:
tool_result = multiply.invoke(result.tool_calls[0])
messages.append(tool_result)
messages

[HumanMessage(content='What is 3 multiplied by 4?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'kf8fhgf81', 'function': {'arguments': '{"a":3,"b":4}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 223, 'total_tokens': 242, 'completion_time': 0.042393343, 'completion_tokens_details': None, 'prompt_time': 0.016905001, 'prompt_tokens_details': None, 'queue_time': 0.059704289, 'total_time': 0.059298344}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e1a99-3b32-7030-815d-50661a640209-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 4}, 'id': 'kf8fhgf81', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 223, 'output_tokens': 19, 'total_tokens': 242}),
 ToolMessage(co

In [131]:
llm_with_tool.invoke(messages).content

''

In [ ]:
#Tool create to get conversion factor from one currency to another
# #Get api key from https://www.exchangerate-api.com/ and replace it in the url in get_conversion_factor tool.
#  
from langchain_core.tools import InjectedToolArg
from typing import Annotated
@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """Returns the conversion factor from base_currency to target_currency."""
    # For demonstration purposes, we'll use hardcoded conversion factors.
    url = f"https://v6.exchangerate-api.com/v6/API_KEY/pair/{base_currency}/{target_currency}"
    response = requests.get(url)
    return response.json()

 

In [133]:
get_conversion_factor.invoke({'base_currency': 'USD', 'target_currency': 'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1778544001,
 'time_last_update_utc': 'Tue, 12 May 2026 00:00:01 +0000',
 'time_next_update_unix': 1778630401,
 'time_next_update_utc': 'Wed, 13 May 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.3751}

In [134]:
#Tool create to convert the base currency to target currency using conversion rate.
#Annotated is used to inject the conversion rate from get_conversion_factor tool to convert tool.

@tool
def convert(base_currency: int , conversion_rate: Annotated[float, InjectedToolArg]) -> float:
    """ this function converts the base currency to target currency using conversion rate."""
    return base_currency * conversion_rate



In [135]:
convert.invoke({'base_currency': 100, 'conversion_rate': 95.3751})    

9537.51

In [136]:
llm = model
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [137]:
messages = [HumanMessage("What is the conversion factor from USD to INR? and based on that can you convert 100 USD to INR currency ?")]
messages

[HumanMessage(content='What is the conversion factor from USD to INR? and based on that can you convert 100 USD to INR currency ?', additional_kwargs={}, response_metadata={})]

In [138]:
ai_message = llm_with_tools.invoke(messages)
messages.append(ai_message)
ai_message.tool_calls



[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'ypzj7a58t',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency': 100},
  'id': 'hpcx43n82',
  'type': 'tool_call'}]

In [139]:
import json

for tool_call in  ai_message.tool_calls:
    #execute the first tool
    if tool_call['name'] == 'get_conversion_factor':
       tool_message1 =  get_conversion_factor.invoke(tool_call)
       print(tool_message1)
       conversion_rate = json.loads(tool_message1.content)['conversion_rate']
       messages.append(tool_message1)
       #inject the conversion rate to convert tool and execute the convert tool

    if tool_call['name'] == 'convert':
        tool_call['args']['conversion_rate'] = conversion_rate
        tool_message2 = convert.invoke(tool_call)
        messages.append(tool_message2)
        

content='{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1778544001, "time_last_update_utc": "Tue, 12 May 2026 00:00:01 +0000", "time_next_update_unix": 1778630401, "time_next_update_utc": "Wed, 13 May 2026 00:00:01 +0000", "base_code": "USD", "target_code": "INR", "conversion_rate": 95.3751}' name='get_conversion_factor' tool_call_id='ypzj7a58t'


In [140]:
messages

[HumanMessage(content='What is the conversion factor from USD to INR? and based on that can you convert 100 USD to INR currency ?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'ypzj7a58t', 'function': {'arguments': '{"base_currency":"USD","target_currency":"INR"}', 'name': 'get_conversion_factor'}, 'type': 'function'}, {'id': 'hpcx43n82', 'function': {'arguments': '{"base_currency":100}', 'name': 'convert'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 323, 'total_tokens': 360, 'completion_time': 0.060928841, 'completion_tokens_details': None, 'prompt_time': 0.023730037, 'prompt_tokens_details': None, 'queue_time': 0.053853572, 'total_time': 0.084658878}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e1a99-47d6-7b81-ab9

In [142]:
llm_with_tools.invoke(messages).content

'The conversion factor from USD to INR is 95.3751. Using this conversion rate, 100 USD is equivalent to 9537.51 INR.'